# 🤖LLM连续对话体验🗨
```
准备工作
1. 到硅基流动用手机号注册个账号，并绑定学校邮箱获得50元赠送余额
https://cloud.siliconflow.cn/
2. 新建API（应用程序接口，Application Programming Interface）密钥
https://cloud.siliconflow.cn/me/account/ak
3. 参考资料: 硅基流动的API手册
https://docs.siliconflow.cn/cn/api-reference/chat-completions/chat-completions#vlm
```
Large Language Model，`input: text`，`output: text`

Available options: `Qwen/Qwen2.5-7B-Instruct`（免费，快，没有深度思考）, `Qwen/Qwen3-8B`（免费，中速，有深度思考）, 其他模型查网页


In [41]:
# 依赖安装
%pip install openai requests tiktoken ipywidgets rich gradio -q

Note: you may need to restart the kernel to use updated packages.


# OpenAI格式的模型服务配置（全局设置）

In [42]:
# OpenAI格式的模型服务配置
from openai import OpenAI
import json

# 配置模型服务
client = OpenAI(
    base_url="https://api.siliconflow.cn/v1",
    # 模型推理服务地址 https://api.siliconflow.cn/v1 是*硅基流动*
    api_key="sk-dmayjpbwtbwhmabpywtvjinhfwxtyxkqijycrsvwanqhriin"    
    # 输入自己的连接密钥 https://cloud.siliconflow.cn/me/account/ak
)

# OpenAI格式的单论对话代码框架

In [43]:
# OpenAI格式的单论对话代码框架

# 先应该是OpenAI格式的模型服务client配置，已经前置设置好了

# 系统提示词，用于指导模型生成内容，可以预设角色、要求等等
system_prompt = "给你物质，你回答化学组成"
# 用户输入，用户的问题
user_question = "可乐"

# 调用模型，创建会话，发送信息
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ],
    temperature=0.7, 
    max_tokens=50,   
    top_p=0.95,        
    frequency_penalty=0,  
    presence_penalty=0   
)

# 输出显示
print(f"系统提示词: {system_prompt}")
print(f"用户输入: {user_question}")
# 从完整回应中提取消息正文内容
ai_answer = response.choices[0].message.content
print(f"\nLLM输出: {ai_answer}")
# 显示获得的完整回应：
print(f"\n完整回应: {json.dumps(response.model_dump(), ensure_ascii=False, indent=2)}")

系统提示词: 给你物质，你回答化学组成
用户输入: 可乐

LLM输出: 可乐的主要成分有水、糖或甜味剂、焦糖色、磷酸、咖啡因、天然和人工香料等。从化学组成的层面来说，可乐中含有水（H2O）、磷酸（H3PO4

完整回应: {
  "id": "01998bf3fb2c564df38cda1b5072ffb2",
  "choices": [
    {
      "finish_reason": "length",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "可乐的主要成分有水、糖或甜味剂、焦糖色、磷酸、咖啡因、天然和人工香料等。从化学组成的层面来说，可乐中含有水（H2O）、磷酸（H3PO4",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1758989646,
  "model": "Qwen/Qwen2.5-7B-Instruct",
  "object": "chat.completion",
  "service_tier": null,
  "system_fingerprint": "",
  "usage": {
    "completion_tokens": 50,
    "prompt_tokens": 22,
    "total_tokens": 72,
    "completion_tokens_details": null,
    "prompt_tokens_details": null
  }
}


# 多轮对话

In [45]:
# 多轮对话
import openai
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

# 先应该是OpenAI格式的模型服务client配置，已经前置设置好了

# 模型配置
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# 模型参数，可以调整：
MODEL_CONFIG = {
    "max_tokens": 200,    # max_tokens 是生成文本的最大长度。
    "top_p": 0.95,         # top_p 是生成文本的随机性。越大，随机性越高。
    "temperature": 0.5,    # temperature 是生成文本的随机性。越大，随机性越高。
}

# 系统提示词
SYSTEM_PROMPT = "你是食谱助手"

# 对话记忆管理
class ChatMemory:
    def __init__(self):
        self.messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    def add_user_message(self, content):
        self.messages.append({"role": "user", "content": content})
    
    def add_assistant_message(self, content):
        self.messages.append({"role": "assistant", "content": content})
    
    def clear_history(self):
        self.messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    def get_messages(self):
        return self.messages

# 全局对话记忆
chat_memory = ChatMemory()

# 创建界面组件
chat_output = widgets.Output()
input_box = widgets.Textarea(
    value='',
    placeholder='请输入您的问题...',
    description='输入:',
    layout=widgets.Layout(width='100%', height='50px')
)
send_button = widgets.Button(description='发送', button_style='primary')
clear_button = widgets.Button(description='清空对话', button_style='warning')

def send_message(b):
    user_input = input_box.value.strip()
    if not user_input:
        return
    
    # 显示用户输入
    with chat_output:
        print(f"用户: {user_input}")
    
    # 清空输入框
    input_box.value = ''
    
    # 添加用户消息到记忆
    chat_memory.add_user_message(user_input)
    
    # 获取AI响应
    try:
        response = ""
        with chat_output:
            print("助手: ", end="")
        
        # 创建流式响应
        stream = client.chat.completions.create(
            model=MODEL_NAME,
            messages=chat_memory.get_messages(),
            stream=True,
            **MODEL_CONFIG
        )
        
        # 流式输出
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                with chat_output:
                    print(content, end="", flush=True)
                response += content
        
        # 添加助手回复到记忆
        chat_memory.add_assistant_message(response)
        
        with chat_output:
            print("\n" + "-" * 50 + "\n")
            
    except Exception as e:
        with chat_output:
            print(f"错误: {str(e)}")

def clear_chat(b):
    chat_memory.clear_history()
    with chat_output:
        clear_output(wait=True)
        print("对话记忆已清空\n" + "=" * 50 + "\n")

# 绑定事件
send_button.on_click(send_message)
clear_button.on_click(clear_chat)

# 回车发送
def on_submit(change):
    if change['new'] and change['new'].endswith('\n'):
        input_box.value = change['new'].rstrip('\n')
        send_message(None)

input_box.observe(on_submit, names='value')

# 显示界面
display(chat_output)
display(widgets.HBox([input_box, send_button, clear_button]))

with chat_output:
    print("对话系统已启动\n" + "=" * 50 + "\n")

Output()

### 🎨Top-p和Temperature区别 

Top-p和Temperature确实都影响生成文本的随机性，但它们的作用层面和机制有本质区别。为了更直观地理解，下表清晰地对比了它们的核心差异：


| 对比维度 | Temperature（温度） | Top-p（核采样） |
|----------|-------------------|-----------------|
| **作用对象** | 整个概率分布的形状 | 候选词的集合范围 |
| **控制逻辑** | 调整所有词概率的平滑度 | 按概率累计动态截断候选列表 |
| **核心机制** | 缩放模型输出的原始分数（logits），然后通过Softmax函数重新计算概率分布 | 从概率最高的词开始累加，直到总和超过阈值p，仅从这个"核心集合"中采样 |
| **直观比喻** | 调色板的温度：控制所有颜色（词汇）的混合均匀程度。温度高，色彩交融更随机；温度低，主体色更突出 | 动态画框：根据画作内容（概率分布）决定一个框的大小，只框定最核心的部分进行创作，忽略边缘杂点 |
| **极端情况** | T→0：退化为贪心搜索，总是选择概率最高的词<br>T→∞：概率分布趋于均匀，近乎随机选择 | p→0：候选集可能非常小，甚至只有一个词，输出确定性高<br>p=1.0：不进行截断，从全部词汇中采样 |
| **主要影响** | 控制随机性的强度。决定模型是"严谨的专家"还是"奔放的艺术家" | 控制候选词的质量底线。确保模型在"合理的选项"内发挥，避免选择离谱的低概率词 |

### 💡 如何选择与配合使用

理解了它们的区别后，在实际应用中可以根据目标来调整这两个参数：

- 追求高度确定性与准确性（如代码生成、事实问答）：建议使用较低的Temperature（如0.1-0.3），并搭配一个中低等的Top-p（如0.7-0.9） 作为安全网，防止输出无关内容。

- 追求平衡与自然（如聊天对话、内容创作）：建议使用中等的Temperature（如0.6-0.8） 并搭配较高的Top-p（如0.9-0.95），这样能在保持连贯性的同时引入足够的多样性。

- 追求最大创造性（如写诗、头脑风暴）：可以尝试较高的Temperature（如0.9-1.2） 和很高的Top-p（如0.95-1.0），让模型充分探索各种可能性。

一个常见的建议是，通常优先调整Temperature来设定随机性的基础水平，然后使用Top-p来精细控制候选词的质量范围。需要注意的是，一般不建议同时将Top-p和Top-k都设置为非默认值，因为它们的功能有重叠，可能会使交互效果复杂化。